In [2]:
import pandas as pd
from pathlib import Path

def process_node_file(input_file, output_file, max_node):
    """
    Filtre le fichier node.csv pour ne conserver que les nœuds <= max_node.
    """
    try:
        df = pd.read_csv(input_file)
        df_reduced = df[df['Node'] <= max_node].copy()
        df_reduced.to_csv(output_file, index=False)
        print(f"✅ Fichier node réduit généré : {output_file}")
    except FileNotFoundError:
        print(f"❌ ERREUR: Fichier {input_file} non trouvé.")
    except Exception as e:
        print(f"Une erreur est survenue lors du traitement de {input_file}: {e}")

def process_od_file(input_file, output_file, max_node):
    """
    Filtre le fichier od.csv pour ne conserver que les paires (O, D)
    où O et D sont <= max_node.
    """
    try:
        df = pd.read_csv(input_file)
        df_reduced = df[(df['O'] <= max_node) & (df['D'] <= max_node)].copy()
        df_reduced.to_csv(output_file, index=False)
        print(f"✅ Fichier OD réduit généré : {output_file}")
    except FileNotFoundError:
        print(f"❌ ERREUR: Fichier {input_file} non trouvé.")
    except Exception as e:
        print(f"Une erreur est survenue lors du traitement de {input_file}: {e}")

def process_net_csv_file(input_file, output_file, max_node):
    """
    Filtre le fichier net.csv pour ne conserver que les liens (A, B)
    où A et B sont <= max_node.
    """
    try:
        df = pd.read_csv(input_file)
        df_reduced = df[(df['A'] <= max_node) & (df['B'] <= max_node)].copy()
        df_reduced.to_csv(output_file, index=False)
        print(f"✅ Fichier net.csv réduit généré : {output_file}")
    except FileNotFoundError:
        print(f"❌ ERREUR: Fichier {input_file} non trouvé.")
    except Exception as e:
        print(f"Une erreur est survenue lors du traitement de {input_file}: {e}")

def process_tntp_file(input_file, output_file, max_node):
    """
    Filtre le fichier .tntp.
    Met à jour les métadonnées et ne conserve que les liens
    où init_node et term_node sont <= max_node.
    """
    try:
        with open(input_file, 'r') as f_in:
            lines = f_in.readlines()

        filtered_data_lines = []
        header_line = None
        in_data_section = False

        for line in lines:
            line_stripped = line.strip()
            if not line_stripped:
                continue

            if line_stripped.startswith('<END OF METADATA>'):
                in_data_section = True
                continue
            
            if not in_data_section:
                continue # Ignore les métadonnées originales
            
            if line_stripped.startswith('~'):
                header_line = line_stripped
                continue
            
            # C'est une ligne de données
            try:
                cols = line_stripped.split('\t')
                init_node = int(cols[0].strip())
                term_node = int(cols[1].strip())
                
                if init_node <= max_node and term_node <= max_node:
                    filtered_data_lines.append(line) # Conserve la ligne originale avec sa mise en forme
            except (ValueError, IndexError):
                print(f"⚠️ Warning: Ligne ignorée dans {input_file}: {line_stripped}")

        # Écrire le nouveau fichier .tntp
        with open(output_file, 'w') as f_out:
            # Écrire les nouvelles métadonnées
            f_out.write(f"<NUMBER OF ZONES> {max_node}\n")
            f_out.write(f"<NUMBER OF NODES> {max_node}\n")
            f_out.write(f"<FIRST THRU NODE> 1\n") # Généralement constant
            f_out.write(f"<NUMBER OF LINKS> {len(filtered_data_lines)}\n")
            f_out.write("<END OF METADATA>\n\n")
            
            if header_line:
                f_out.write(f"{header_line}\n")
            
            f_out.writelines(filtered_data_lines)
            
        print(f"✅ Fichier .tntp réduit généré : {output_file}")

    except FileNotFoundError:
        print(f"❌ ERREUR: Fichier {input_file} non trouvé.")
    except Exception as e:
        print(f"Une erreur est survenue lors du traitement de {input_file}: {e}")

#
# -----------------------------------------------------------
# ▼▼▼ SEULE LA FONCTION CI-DESSOUS A ÉTÉ MODIFIÉE ▼▼▼
# -----------------------------------------------------------
#
def generate_reduced_network(max_node_id):
    """
    Fonction principale pour générer tous les fichiers réduits
    pour un nombre maximal de nœuds donné.
    """
    print(f"--- Génération du réseau réduit pour les nœuds 1 à {max_node_id} ---")
    
    # Définir les chemins
    # Répertoire d'ENTRÉE : ../sioux_falls (par rapport au script)
    input_dir = Path("../sioux_falls")
    
    # Répertoire de SORTIE : sioux_falls_reduced_1_to_XX (dans le dossier du script)
    output_dir_name = f"sioux_falls_reduced_1_to_{max_node_id}"
    output_dir = Path.cwd() / output_dir_name
    
    # Créer le dossier de sortie
    output_dir.mkdir(exist_ok=True)
    
    # Définir les noms de fichiers
    files = {
        "node": "SiouxFalls_node.csv",
        "net_csv": "SiouxFalls_net.csv",
        "net_tntp": "SiouxFalls_net.tntp",
        "od": "SiouxFalls_od.csv"
    }

    # Lancer les processus de réduction
    process_node_file(
        input_dir / files["node"],  # Chemin d'entrée mis à jour
        output_dir / f"reduced_{files['node']}", # Chemin de sortie mis à jour
        max_node_id
    )
    
    process_od_file(
        input_dir / files["od"], # Chemin d'entrée mis à jour
        output_dir / f"reduced_{files['od']}", # Chemin de sortie mis à jour
        max_node_id
    )
    
    process_net_csv_file(
        input_dir / files["net_csv"], # Chemin d'entrée mis à jour
        output_dir / f"reduced_{files['net_csv']}", # Chemin de sortie mis à jour
        max_node_id
    )
    
    process_tntp_file(
        input_dir / files["net_tntp"], # Chemin d'entrée mis à jour
        output_dir / f"reduced_{files['net_tntp']}", # Chemin de sortie mis à jour
        max_node_id
    )
    
    print(f"--- Génération terminée. Fichiers disponibles dans: {output_dir} ---")

# --- COMMENT UTILISER LE SCRIPT ---
if __name__ == "__main__":
    
    # ⬇️⬇️ MODIFIEZ CETTE VALEUR ⬇️⬇️
    # Indiquez le numéro du dernier nœud à conserver.
    FINAL_NODE_INDICE = 14 
    
    generate_reduced_network(FINAL_NODE_INDICE)

--- Génération du réseau réduit pour les nœuds 1 à 14 ---
✅ Fichier node réduit généré : c:\Users\lerouxb\Documents\Code\lam-traffic-assignment\data\sioux_falls_reduced\sioux_falls_reduced_1_to_14\reduced_SiouxFalls_node.csv
✅ Fichier OD réduit généré : c:\Users\lerouxb\Documents\Code\lam-traffic-assignment\data\sioux_falls_reduced\sioux_falls_reduced_1_to_14\reduced_SiouxFalls_od.csv
✅ Fichier net.csv réduit généré : c:\Users\lerouxb\Documents\Code\lam-traffic-assignment\data\sioux_falls_reduced\sioux_falls_reduced_1_to_14\reduced_SiouxFalls_net.csv
✅ Fichier .tntp réduit généré : c:\Users\lerouxb\Documents\Code\lam-traffic-assignment\data\sioux_falls_reduced\sioux_falls_reduced_1_to_14\reduced_SiouxFalls_net.tntp
--- Génération terminée. Fichiers disponibles dans: c:\Users\lerouxb\Documents\Code\lam-traffic-assignment\data\sioux_falls_reduced\sioux_falls_reduced_1_to_14 ---
